# Game Store Rental Management System

## 25COA122 Coursework Information

Notebook showing that **feedbackManager.pyc** and **subscriptionManager.pyc** work.

In [12]:
!python --version


Python 3.12.12


In [4]:
# Upload the pyc file.
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [24]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import datetime as daTi
import matplotlib.pyplot as plt

gDate = daTi.datetime.today().strftime('%Y-%m-%d')

# Check Feedback Manager Works.
feedback_list = fmSL.load_feedback()
print(feedback_list)  # Should print the list of feedback records.
fmSL.add_feedback("dft01", 4, "Not bad", "Game_Feedback.txt")

# Check Subscription Manager Works.
subscriptions = smSL.load_subscriptions()
print(smSL.check_subscription("lbro", subscriptions))
# Should return True or False based on the current date.
print(smSL.get_rental_limit("Basic"))  # Should return 2.
print(smSL.get_rental_limit("Premium"))  # Should return 7.



[{'GameID': 'Art01', 'Rating': 4, 'Comments': 'Hilarious!'}, {'GameID': 'bac02', 'Rating': 2, 'Comments': 'Not exciting'}, {'GameID': 'chs03', 'Rating': 5, 'Comments': 'My favourite game of all time'}]
True
2
7


# **Database**

In [45]:
#read file
def readFile(szFileName):
  rows = []
  try:
    with open(szFileName, "r") as file:
      for line in file:
        line = line.strip()
        if line != "":
          rows.append(line.split(","))
  except FileNotFoundError:
    print("'%s' file not found" %(szFileName))
  return rows

#save to file
def saveToFile(rows, szFileName):
  try:
    with open(szFileName, "w") as file:
      for innerRow in rows:
        newLine = ""
        for word in innerRow:
          newLine += word + ","
        newLine = newLine[:-1]
        #since its the last element, the comma isnt needed
        newLine += "\n"
        file.write(newLine)
      print("'%s' file has been saved" %(szFileName))
  except FileNotFoundError:
    print("'%s' file not found" %(szFileName))

#is customer id valid
def isValidCust(szId):
  customers = readFile("Subscription_Info.txt")
  for customer in customers:
    if customer[0] == szId:
      return True

  return False

def getCustomer(szCustomerId): #returns their subscription
  customers = readFile("Subscription_Info.txt")
  for customer in customers:
    if customer[0] == szCustomerId:
      return customer

#is game id valid
def isValidGame(szId,cType):
  #cType: V for video game, B for board game
  if cType == "V":
    szFileName = "Video_Game_Info.txt"
  else:
    szFileName = "Board_Game_Info.txt"

  games = readFile(szFileName)
  for game in games:
    if game[0] == szId:
      return True

  #print("Game Id '%s' isnt valid" %(szId))
  return False


#is game available
def isGameAvailable(szGameId):
  if isValidGame(szGameId, "V") or isValidGame(szGameId, "B"):
    rentals = readFile("Rental.txt")

    for rental in rentals:
      szRentalGameId = rental[0]
      returnDate = rental[2]

      if returnDate == "" and szRentalGameId == szGameId:
        return False
  else:
    #print("Unable to check an invalid game ID")
    return False
  return True

def getGameName(szGameId):
  if isValidGame(szGameId, "V") or isValidGame(szGameId, "B"):
    games = readFile("Video_Game_Info.txt") + readFile("Board_Game_Info.txt")
    for game in games:
      if game[0] == szGameId:
        return game[1]

def addSubscription(szCustomerId, szType, startDate, endDate, subscriptions):
  if len(szCustomerId) != 4 or (not szCustomerId.isalpha()):
    return "Invalid customer id. Must be exactly 4 letters"
  if isValidCust(szCustomerId):
    return "Customer Id '%s' already exists" %(szCustomerId)

  startDate = daTi.datetime.strptime(startDate, "%Y-%m-%d")
  endDate = daTi.datetime.strptime(endDate, "%Y-%m-%d")
  subscriptions[szCustomerId] = {
    "SubscriptionType": szType,
    "StartDate": startDate,
    "EndDate": endDate
    }

  #rewrite subscriptiosnfile
  #since its a map dont use the save to file function
  with open("Subscription_Info.txt", "w") as file:
    file.write("CustomerID,SubscriptionType,StartDate,EndDate\n")

    for szCustomerId, subscription in subscriptions.items():
      szType = subscription["SubscriptionType"]
      startDate = subscription["StartDate"].strftime("%Y-%m-%d")
      endDate = subscription["EndDate"].strftime("%Y-%m-%d")
      newLine = szCustomerId + "," + szType + "," + startDate + "," + endDate + "\n"
      file.write(newLine)

  #reload subscriptions
  subscriptions = smSL.load_subscriptions()

  return "Subscription added"


# **Game Search**

In [47]:
def searchGame(szSearch):
  results = []
  szSearch = szSearch.lower()

  games = readFile("Video_Game_Info.txt") + readFile("Board_Game_Info.txt")
  for game in games:
    szGameId = game[0]
    szName = game[1].lower()

    if szSearch in szName:
      results.append(game)
  return results

def displaySearch(results):
  if results == []:
    print("No search results")
  else:
    #display results formatted
    print("Search Results:")
    for result in results:
      print("Game ID: %s Name: %s" %(result[0], result[1]))



# **Game Rent**

In [27]:
def getRentals(szCustomerId):
  allRentals = readFile("Rental.txt")
  custRentals = []

  for rental in allRentals:
    if rental[2] == "" and rental[3] == szCustomerId:
      custRentals.append(rental)

  return custRentals

def rentGame(szCustomerId, szGameId):
  #check if customer has a subscription
  bCust = smSL.check_subscription(szCustomerId, subscriptions)

  if not bCust:
    return "Customer Not Found"

  #get the customers subscription details
  customer = getCustomer(szCustomerId)
  szType = customer[1]
  startDate = customer[2]
  endDate = customer[3]

  #check game id is valid
  bGame = isValidGame(szGameId, "V") or isValidGame(szGameId, "B")

  if bGame:
    #check game is available
    bAvailable = isGameAvailable(szGameId)
    if bAvailable:
      #get rental limit and check how many current rentals the customer has
      iRentalLimit = smSL.get_rental_limit(szType)
      allRentals = readFile("Rental.txt")
      rentals = getRentals(szCustomerId)
      iCurrentRentals = len(rentals)

      if iRentalLimit == iCurrentRentals:
        return False
      else:
        newRental = [szGameId, gDate, "", szCustomerId]
        allRentals.append(newRental)
        saveToFile(allRentals, "Rental.txt")
        return True
    else:
      print("Game is not available")
      return False
  else:
    print("Game Id '%s' isnt valid" %(szGameId))
    return False






# **Game Return**

In [28]:
def isValidRating(iRating):
  if iRating >= 1 and iRating <= 5:
    return True
  else:
    return False

def returnGame(szGameId, iRating, szComment):
  #check game id is valid
  bGame = isValidGame(szGameId, "V") or isValidGame(szGameId, "B")
  if bGame:
    #if game is available then it hasnt been rented out
    bAvailable = isGameAvailable(szGameId)
    if not bAvailable:
      allRentals = readFile("Rental.txt")
      for rental in allRentals:
        if rental[0] == szGameId and rental[2] == "":
          rental[2] = gDate
          saveToFile(allRentals, "Rental.txt")
          fmSL.add_feedback(szGameId, iRating, szComment, "Game_Feedback.txt")
          print("Game has been returned")
          return True
    else:
      print("Game hasnt been rented out")
  else:
    print("Game Id '%s' isnt valid" %(szGameId))

  return

# **Booking**

In [49]:
def isValidTimeSlot(szTimeSlot):
  if szTimeSlot == "2pm" or szTimeSlot == "6pm":
    return True
  else:
    return False

def countPeopleInSlot(szDate, szTimeSlot):
  bookings = readFile("Booking.txt")
  iCount = 0

  for booking in bookings:
    if booking[1] == szDate and booking[2] == szTimeSlot:
      iPeople = int(booking[3]) + 1
      iCount += iPeople

  return iCount

def hasExistingBooking(szCustomerId, szDate, szTimeSlot):
  bookings = readFile("Booking.txt")

  for booking in bookings:
    if booking[0] == szCustomerId and booking[1] == szDate and booking[2] == szTimeSlot:
      return True

def bookSession(szCustomerId, szDate, szTimeSlot, guests):
  if not isValidCust(szCustomerId):
    print("Customer Id '%s' isnt valid" %(szCustomerId))
    return False
  elif not smSL.check_subscription(szCustomerId, subscriptions):
    print("Customer Id '%s' has no subscription" %(szCustomerId))
    return False
  elif not isValidTimeSlot(szTimeSlot):
    print("Time slot '%s' isnt valid" %(szTimeSlot))
    return False
  elif guests < 0 or guests > 3:
    print("Number of guests must be between 0 and 3")
    return False
  elif hasExistingBooking(szCustomerId, szDate, szTimeSlot):
    print("Customer has an existing booking")
    return False
  elif (countPeopleInSlot(szDate, szTimeSlot) + 1 + guests) > 50:
    print("Too many people in slot")
    return False
  else:
    newBooking = [szCustomerId, szDate, str(szTimeSlot), str(guests)]
    bookings = readFile("Booking.txt")
    bookings.append(newBooking)
    saveToFile(bookings, "Booking.txt")





# Inventory **Pruning**

In [51]:
#get all game ids
def getGameIds():
  videoGames = readFile("Video_Game_Info.txt")
  boardGames = readFile("Board_Game_Info.txt")
  games = videoGames + boardGames
  gameIds = []

  for game in games:
    gameIds.append(game[0])

  return gameIds

#Get total number of rentals for each game
#Lower the total number of rentals the more unpopular

#make a dictionary which maps game id to total rentals
def getRentalCount():
  rentals = readFile("Rental.txt")
  allGameIds = getGameIds()
  #add all the game ids into the dictionary with a default value 0
  rentalCounts = {szGameId: 0 for szGameId in allGameIds}

  for rental in rentals:
    szGameId = rental[0]
    rentalCounts[szGameId] += 1

  return rentalCounts

def getUnpopularGames():
  rentalCounts = getRentalCount()
  unpopularGames = []
  MIN_RENTAL_COUNT = 1

  for szGameId, iRentalCount in rentalCounts.items():
    if iRentalCount <= MIN_RENTAL_COUNT:
      unpopularGames.append((szGameId,iRentalCount))

  return unpopularGames

def displayRemovalSuggestions(unpopularGames):
  if not unpopularGames:
    print("There are no unpopular games")
    return

  print("Suggested games to be removed:")
  for szGameId, iRentalCount in unpopularGames:
    szGameName = getGameName(szGameId)
    print("Game ID: %s Name: %s Rentals: %d" %(szGameId, szGameName, iRentalCount))


#visualisation
def plotRentalFreq():
  rentalCounts = getRentalCount()
  gameIds = list(rentalCounts.keys())
  rentalCounts = list(rentalCounts.values())

  plt.figure()
  plt.bar(gameIds, rentalCounts)
  plt.xlabel("Game ID")
  plt.ylabel("Rental Count")
  plt.title("Rental Frequency of Games")
  plt.xticks(rotation=90)
  plt.tight_layout()
  plt.show()

# **MENU**

In [54]:
import ipywidgets as widgets
from datetime import datetime

output = widgets.Output()


#search

search_text = widgets.Text(description="Search:")
search_button = widgets.Button(description="Search")

def on_search_clicked(b):
  output.clear_output()
  with output:
    results = searchGame(search_text.value)
    displaySearch(results)

search_button.on_click(on_search_clicked)


# register (DatePicker)

register_customer_id = widgets.Text(description="Customer ID:")

subscription_type = widgets.Dropdown(
  options=["Basic", "Premium"],
  description="Type:"
)

start_date_picker = widgets.DatePicker(
  description="Start Date:"
)

subscription_years = widgets.IntSlider(
  value=1,
  min=1,
  max=5,
  step=1,
  description="Years:"
)

register_button = widgets.Button(description="Register Customer")

def on_register_clicked(b):
  output.clear_output()
  with output:
    if start_date_picker.value is None:
      print("Please select a start date.")
      return

    # DatePicker returns a datetime.date -> convert to datetime at midnight
    start_dt = datetime.combine(start_date_picker.value, datetime.min.time())

    years = subscription_years.value
    try:
      end_dt = start_dt.replace(year=start_dt.year + years)
    except ValueError:
      # Very rare edge case: Feb 29 -> non-leap year
      # Move to Feb 28 if needed
      if start_dt.month == 2 and start_dt.day == 29:
        start_dt = start_dt.replace(day=28)
        end_dt = start_dt.replace(year=start_dt.year + years)

    result = addSubscription(
      register_customer_id.value,
      subscription_type.value,
      start_dt.strftime("%Y-%m-%d"),
      end_dt.strftime("%Y-%m-%d"),
      subscriptions
    )
    print(result)

register_button.on_click(on_register_clicked)


#rent

rent_customer = widgets.Text(description="Customer ID:")
rent_game_id = widgets.Text(description="Game ID:")
rent_button = widgets.Button(description="Rent Game")

def on_rent_clicked(b):
  output.clear_output()
  with output:
    rentGame(rent_customer.value, rent_game_id.value)


rent_button.on_click(on_rent_clicked)


#return

return_game_id = widgets.Text(description="Game ID:")

rating_dropdown = widgets.Dropdown(
  options=[1, 2, 3, 4, 5],
  description="Rating:"
)

comment_box = widgets.Textarea(
  description="Comment:",
  placeholder="Optional"
)

return_button = widgets.Button(description="Return Game")

def on_return_clicked(b):
  output.clear_output()
  with output:
    returnGame(
      return_game_id.value,
      rating_dropdown.value,
      comment_box.value
    )

return_button.on_click(on_return_clicked)


#booking (DatePicker)

book_customer = widgets.Text(description="Customer ID:")

book_date_picker = widgets.DatePicker(
  description="Date:"
)

book_slot = widgets.Dropdown(
  options=["2pm", "6pm"],
  description="Time:"
)

book_guests = widgets.IntSlider(
  value=0,
  min=0,
  max=3,
  description="Guests:"
)

book_button = widgets.Button(description="Book Session")

def on_book_clicked(b):
  output.clear_output()
  with output:
    if book_date_picker.value is None:
      print("Please select a booking date.")
      return

    booking_date_str = book_date_picker.value.strftime("%Y-%m-%d")

    bookSession(
      book_customer.value,
      booking_date_str,
      book_slot.value,
      book_guests.value
    )

book_button.on_click(on_book_clicked)


#pruning

prune_button = widgets.Button(description="Show Inventory Pruning Suggestions")

def on_prune_clicked(b):
  output.clear_output()
  with output:
    plotRentalFreq()
    unpopular = getUnpopularGames()
    displayRemovalSuggestions(unpopular)

prune_button.on_click(on_prune_clicked)


#display menu
menu = widgets.VBox([
  widgets.HTML("<b><h1>Games Store Management System<></b>"),
  widgets.HTML("<hr>"),

  widgets.Label("Register Customer"),
  register_customer_id,
  subscription_type,
  start_date_picker,
  subscription_years,
  register_button,

  widgets.HTML("<hr>"),

  search_text,
  search_button,

  widgets.HTML("<hr>"),

  rent_customer,
  rent_game_id,
  rent_button,

  widgets.HTML("<hr>"),

  return_game_id,
  rating_dropdown,
  comment_box,
  return_button,

  widgets.HTML("<hr>"),

  book_customer,
  book_date_picker,
  book_slot,
  book_guests,
  book_button,

  widgets.HTML("<hr>"),

  prune_button,
  output
])

display(menu)


Student ID = F534874
# **Documentation**

**Database:**
This cell contains functions for reading, writing text files and for checking customers, games, rentals and subscriptions in the system.

```
readFile() - reads a text file and returns its contents as a list of rows

saveToFile() - writes a list of rows to a text file

isValidCust() - checks whether a customer id exists

getCustomer() - returns the customer record for a give customer id

isValidGame() - checks whether a game ID exists in any of the game files

isGameAvailable() - checks if a game is available to rent

addSubscription() - creates a new customer and adds its subscription to the subscription file
```
**Game Search:**
This cell contains functions for searching games by name and displaying the search results.
```
searchGame() - searches video and board games for matching names and returns the results

displaySearch() - displays the search results or prints a message if no results are found
```
**Game Rent:**
This cell contains functions for retrieving a customer's current rentals and for renting a game to a customer.
```
getRentals() - returns a list of games currently rented by a specific customer

rentGame() - checks customer validity, subscription status, rental limits, and game availability before renting a game
```
**Game Return:**
This cell contains functions for validating game ratings and for returning a rented game with feedback.
```
isValidRating() - checks whether a rating is between 1 and 5

returnGame() - returns a rented game, updates the rental file, and records customer feedback
```
**Booking:**
This cell contains functions for validating time slots and managing customer session bookings.
```
isValidTimeSlot() - checks whether a time slot is valid

countPeopleInSlot() - counts how many people are booked into a specific date and time slot

hasExistingBooking() - checks whether a customer already has a booking for a given date and time slot

bookSession() - validates booking rules and creates a new session booking
```
**Inventory Pruning:**
This cell contains functions for analysing game rental data to identify unpopular games and suggest removals from inventory.
```
getGameIds() - returns a list of all game IDs from both video and board game files

getRentalCount() - counts the total number of rentals for each game

getUnpopularGames() - identifies games with low rental counts

displayRemovalSuggestions() - displays games suggested for removal with their rental counts

plotRentalFreq() - displays a bar chart showing rental frequency for each game
```







